<a href="https://colab.research.google.com/github/angePatrickKOUTOU/MesAgentIA/blob/main/AgentIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q crewai crewai-tools pypdf

In [2]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool
print("CrewAI OK ✅")

CrewAI OK ✅


In [3]:
import os
from google.colab import userdata

for k in ["GEMINI_API_KEY", "SERPER_API_KEY", "GMAIL_USER", "GMAIL_APP_PASSWORD"]:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        print(f"⚠️ Secret manquant : {k}")



In [4]:
from google.colab import files
from pypdf import PdfReader

up = files.upload()  # envoie ton CV en PDF
nom = list(up.keys())[0]
cv_texte = "\n".join(p.extract_text() for p in PdfReader(nom).pages)
print(cv_texte[:500])

Saving cv_Patrick_data_analyst.pdf to cv_Patrick_data_analyst (3).pdf
ANGE PATRICK KOUTOU
STAGE DE 6 MOIS – DATA ANALYST
+33-7-53-92-50-24 | Koutoupatrick@gmail.com | Permis B
France – Mobile sur toute la France
RECHERCHE : Stage de 6 mois à partir de février 2027
Compétences techniques
•
Visualisation & BI : Power BI
,
tableaux de bord
,
KPI
, aide à la dé-
cision.
•
Requêtage : 
SQL
, PostgreSQL,
MySQL, modélisation.
•
ETL :
Talend, Datawarehouse, nettoy-
age et transformation des données.
•
Analyse & reporting : Excel
,
VBA
,
Python
, génération automatisée de 


In [5]:
import smtplib
from email.message import EmailMessage
from crewai.tools import tool
from crewai_tools import SerperDevTool

DRY_RUN = True  # passe à False seulement quand tu es sûr du résultat

recherche_web = SerperDevTool()

@tool("Envoyer une candidature par email")
def envoyer_candidature(destinataire: str, sujet: str, corps: str) -> str:
    """Envoie un email de candidature. Arguments : destinataire, sujet, corps."""
    if DRY_RUN:
        print(f"\n--- [TEST] Email à {destinataire} ---\nSujet : {sujet}\n\n{corps}\n")
        return "Mode test : email affiché, non envoyé."
    msg = EmailMessage()
    msg["From"] = os.environ["GMAIL_USER"]
    msg["To"] = destinataire
    msg["Subject"] = sujet
    msg.set_content(corps)
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
        s.login(os.environ["GMAIL_USER"], os.environ["GMAIL_APP_PASSWORD"])
        s.send_message(msg)
    return f"Email envoyé à {destinataire}."

In [6]:
from crewai import Agent, Task, Crew, Process, LLM

# Mise à jour du modèle vers gemini-1.5-flash pour corriger l'erreur d'indisponibilité 404
llm = LLM(model="gemini/gemini-1.5-flash",
          api_key=os.environ["GEMINI_API_KEY"], temperature=0.3)

chercheur = Agent(
    role="Chercheur d'offres de stage en data",
    goal=("Trouver des offres de stage de 6 mois en data analyse et "
          "développement data qui correspondent vraiment au CV du candidat"),
    backstory=(
        "Tu es un expert du recrutement tech en France. Tu connais les "
        "intitulés équivalents : Data Analyst, Business Intelligence, "
        "Analytics Engineer, Data Engineer, Développeur Data, "
        "Data Scientist junior, Reporting/Power BI. Tu écartes les offres "
        "qui ne sont pas des stages, qui durent moins de 4 mois, ou qui "
        "exigent plus d'expérience qu'un étudiant n'en a."
    ),
    tools=[recherche_web], llm=llm, verbose=True,
)

candidat = Agent(
    role="Rédacteur de candidatures de stage en data",
    goal="Rédiger des emails de candidature courts, personnalisés et honnêtes",
    backstory=(
        "Tu écris en français professionnel pour un étudiant en informatique "
        "qui cherche un stage de 6 mois en data. Tu mets en avant les "
        "compétences du CV qui répondent à l'offre (SQL, Python, "
        "visualisation, ETL, etc.) et tu n'inventes JAMAIS une compétence, "
        "un projet ou une expérience absents du CV."
    ),
    tools=[envoyer_candidature], llm=llm, verbose=True,
)

tache_recherche = Task(
    description=(
        "Voici le CV du candidat :\n{cv}\n\n"
        "Recherche 8 offres de STAGE de {duree} pour les postes suivants : "
        "{postes}. Lieu : {lieu}. Période de début souhaitée : {debut}.\n\n"
        "Fais plusieurs recherches avec des mots-clés différents "
        "(ex. « stage data analyst 6 mois », « stage data engineer », "
        "« stage business intelligence », « stage développeur data »).\n\n"
        "Pour chaque offre, donne : titre, entreprise, lieu, durée, lien, "
        "email de contact s'il existe, les compétences demandées, et un "
        "score de correspondance de 1 à 5 avec le CV, justifié en une phrase. "
        "Garde les 5 meilleures offres, triées par score."
    ),
    expected_output=("Liste des 5 meilleures offres au format structuré, "
                     "avec score de correspondance."),
    agent=chercheur,
)

tache_candidature = Task(
    description=(
        "Pour chacune des offres retenues, rédige une candidature basée "
        "uniquement sur ce CV :\n{cv}\n\n"
        "Pour chaque offre :\n"
        "1. Un objet clair : « Candidature stage {duree} - <poste> ».\n"
        "2. Un email de 8 à 10 lignes maximum : accroche liée à l'entreprise, "
        "2 ou 3 compétences ou projets du CV qui répondent à l'offre, "
        "disponibilité {debut}, mobilité sur toute la France, "
        "formule de politesse.\n"
        "3. Si l'offre a un email de contact, envoie-le avec l'outil. "
        "Sinon, fournis le texte prêt à copier avec le lien de candidature."
    ),
    expected_output=("Récapitulatif par offre : email envoyé ou texte à copier, "
                     "avec le lien."),
    agent=candidat,
    context=[tache_recherche],
    human_input=True,  # tu valides avant la finalisation
)

equipe = Crew(
    agents=[chercheur, candidat],
    tasks=[tache_recherche, tache_candidature],
    process=Process.sequential,
)

In [7]:
import nest_asyncio
import os
from crewai import Agent, Task, Crew, Process, LLM

nest_asyncio.apply()

# Utilisation de la version spécifique et stable reconnue par l'API v1beta avec le préfixe de fournisseur
llm_correct = LLM(model="gemini/gemini-1.5-flash-001",
                  api_key=os.environ["GEMINI_API_KEY"], temperature=0.3)

chercheur.llm = llm_correct
candidat.llm = llm_correct

equipe_corrigee = Crew(
    agents=[chercheur, candidat],
    tasks=[tache_recherche, tache_candidature],
    process=Process.sequential,
)

# Lancement asynchrone avec la configuration mise à jour
resultat = await equipe_corrigee.kickoff_async(inputs={
    "cv": cv_texte,
    "recherche": "stage informatique 6 mois",
    "lieu": "France",
    "duree": "6 mois",
    "postes": "Data Analyst, Business Intelligence, Data Engineer, Développeur Data",
    "debut": "février 2027"
})
print(resultat)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chercheur d'offres de stage en data                                                                     │
│                                                                                                                 │
│  Task: Voici le CV du candidat :                                                                                │
│  ANGE PATRICK KOUTOU                                                                                            │
│  STAGE DE 6 MOIS – DATA ANALYST                                                                                 │
│  +33-7-53-92-50-24 | Koutoupatrick@gmail.com | Permis B                                                         │
│  France – Mobile sur toute la France                                                                            │
│  RECHERCHE : Stage de 6 mois à partir de février 2027                                                           │
│  Compétences techniques                                                                                         │
│  •                                                                                                              │
│  Visualisation & BI : Power BI                                                                                  │
│  ,                                                                                                              │
│  tableaux de bord                                                                                               │
│  ,                                                                                                              │
│  KPI                                                                                                            │
│  , aide à la dé-                                                                                                │
│  cision.                                                                                                        │
│  •                                                                                                              │
│  Requêtage :                                                                                                    │
│  SQL                                                                                                            │
│  , PostgreSQL,                                                                                                  │
│  MySQL, modélisation.                                                                                           │
│  •                                                                                                              │
│  ETL :                                                                                                          │
│  Talend, Datawarehouse, nettoy-                                                                                 │
│  age et transformation des données.                                                                             │
│  •                                                                                                              │
│  Analyse & reporting : Excel                                                                                    │
│  ,                                                                                                              │
│  VBA                                                                                                            │
│  ,                                                                                                              │
│  Python                                                                                                         │
│  , génération automatisée de rap-                      

ERROR:root:Google Gemini API error: 404 - models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


An unknown error occurred. Please check the details below.
Error details: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
An unknown error occurred. Please check the details below.
Error details: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chercheur d'offres de stage en data                                                                     │
│                                                                                                                 │
│  Task: Voici le CV du candidat :                                                                                │
│  ANGE PATRICK KOUTOU                                                                                            │
│  STAGE DE 6 MOIS – DATA ANALYST                                                                                 │
│  +33-7-53-92-50-24 | Koutoupatrick@gmail.com | Permis B                                                         │
│  France – Mobile sur toute la France                                                                            │
│  RECHERCHE : Stage de 6 mois à partir de février 2027                                                           │
│  Compétences techniques                                                                                         │
│  •                                                                                                              │
│  Visualisation & BI : Power BI                                                                                  │
│  ,                                                                                                              │
│  tableaux de bord                                                                                               │
│  ,                                                                                                              │
│  KPI                                                                                                            │
│  , aide à la dé-                                                                                                │
│  cision.                                                                                                        │
│  •                                                                                                              │
│  Requêtage :                                                                                                    │
│  SQL                                                                                                            │
│  , PostgreSQL,                                                                                                  │
│  MySQL, modélisation.                                                                                           │
│  •                                                                                                              │
│  ETL :                                                                                                          │
│  Talend, Datawarehouse, nettoy-                                                                                 │
│  age et transformation des données.                                                                             │
│  •                                                                                                              │
│  Analyse & reporting : Excel                                                                                    │
│  ,                                                                                                              │
│  VBA                                                                                                            │
│  ,                                                                                                              │
│  Python                                                                                                         │
│  , génération automatisée de rap-                      

ERROR:root:Google Gemini API error: 404 - models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


An unknown error occurred. Please check the details below.
Error details: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
An unknown error occurred. Please check the details below.
Error details: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Chercheur d'offres de stage en data                                                                     │
│                                                                                                                 │
│  Task: Voici le CV du candidat :                                                                                │
│  ANGE PATRICK KOUTOU                                                                                            │
│  STAGE DE 6 MOIS – DATA ANALYST                                                                                 │
│  +33-7-53-92-50-24 | Koutoupatrick@gmail.com | Permis B                                                         │
│  France – Mobile sur toute la France                                                                            │
│  RECHERCHE : Stage de 6 mois à partir de février 2027                                                           │
│  Compétences techniques                                                                                         │
│  •                                                                                                              │
│  Visualisation & BI : Power BI                                                                                  │
│  ,                                                                                                              │
│  tableaux de bord                                                                                               │
│  ,                                                                                                              │
│  KPI                                                                                                            │
│  , aide à la dé-                                                                                                │
│  cision.                                                                                                        │
│  •                                                                                                              │
│  Requêtage :                                                                                                    │
│  SQL                                                                                                            │
│  , PostgreSQL,                                                                                                  │
│  MySQL, modélisation.                                                                                           │
│  •                                                                                                              │
│  ETL :                                                                                                          │
│  Talend, Datawarehouse, nettoy-                                                                                 │
│  age et transformation des données.                                                                             │
│  •                                                                                                              │
│  Analyse & reporting : Excel                                                                                    │
│  ,                                                                                                              │
│  VBA                                                                                                            │
│  ,                                                                                                              │
│  Python                                                                                                         │
│  , génération automatisée de rap-                      

ERROR:root:Google Gemini API error: 404 - models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.
ERROR:crewai.flow.runtime:Error executing listener call_llm_native_tools: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


An unknown error occurred. Please check the details below.
Error details: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}
An unknown error occurred. Please check the details below.
Error details: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash-001 is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}